# 05 Final Load Prep

Use this notebook to compute final KPIs, prepare the Tableau-ready dataset, and export the exact file used in the dashboard.

In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()

In [ ]:
DATA_PATH = PROJECT_ROOT / 'data/processed/Beijing PM25 Data Cleaned.csv'
TABLEAU_READY_PATH = PROJECT_ROOT / 'data/processed/tableau_ready_dataset.csv'

df = pd.read_csv(DATA_PATH)
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
df.head()

In [ ]:

# Dashboard 1 — Temporal Patterns

# 1. % Hours Above 150 µg/m³ by Month
above_150 = df.groupby('month')['pm2_5_cleaned'].apply(lambda x: (x > 150).mean() * 100)
pct_above_150_by_month = above_150.reset_index(name='pct_hours_above_150')
pct_above_150_by_month.to_csv(PROJECT_ROOT / 'data/processed/pct_above_150_by_month.csv', index=False)

# 2. Average PM2.5 by Hour of Day
avg_pm25_by_hour = df.groupby('hour')['pm2_5_cleaned'].mean().reset_index(name='avg_pm25')
avg_pm25_by_hour.to_csv(PROJECT_ROOT / 'data/processed/avg_pm25_by_hour.csv', index=False)

# 3. Hour × Month Heatmap
hour_month_heatmap = df.groupby(['hour', 'month'])['pm2_5_cleaned'].mean().reset_index(name='avg_pm25')
hour_month_heatmap.to_csv(PROJECT_ROOT / 'data/processed/hour_month_heatmap.csv', index=False)

# 4. Monthly Average PM2.5 vs WHO Limit
monthly_avg_pm25 = df.groupby('month')['pm2_5_cleaned'].mean().reset_index(name='avg_pm25')
monthly_avg_pm25['who_limit'] = 15  # WHO annual limit, but using for monthly comparison
monthly_avg_pm25.to_csv(PROJECT_ROOT / 'data/processed/monthly_avg_pm25_vs_who.csv', index=False)



# Dashboard 2 — Risk & Severity Overview

# 1. Average PM2.5 Concentration (overall)
overall_avg_pm25 = pd.DataFrame({'overall_avg_pm25': [df['pm2_5_cleaned'].mean()]})
overall_avg_pm25.to_csv(PROJECT_ROOT / 'data/processed/overall_avg_pm25.csv', index=False)

# 2. Hazardous Days per Year
df['date'] = df['datetime'].dt.date
daily_avg = df.groupby(['year', 'date'])['pm2_5_cleaned'].mean().reset_index()
daily_avg['hazardous'] = daily_avg['pm2_5_cleaned'] > 150
hazardous_days_per_year = daily_avg.groupby('year')['hazardous'].sum().reset_index(name='hazardous_days')
hazardous_days_per_year.to_csv(PROJECT_ROOT / 'data/processed/hazardous_days_per_year.csv', index=False)

# 3. AQI Category Breakdown (% hours in each category)
def aqi_category(pm):
    if pm <= 12: return 'Good'
    elif pm <= 35.4: return 'Moderate'
    elif pm <= 55.4: return 'Unhealthy (Sensitive)'
    elif pm <= 150.4: return 'Unhealthy'
    elif pm <= 250.4: return 'Very Unhealthy'
    else: return 'Hazardous'

df['aqi_cat'] = df['pm2_5_cleaned'].apply(aqi_category)
aqi_breakdown = df['aqi_cat'].value_counts(normalize=True).reset_index()
aqi_breakdown.columns = ['aqi_category', 'pct_hours']
aqi_breakdown['pct_hours'] *= 100
aqi_breakdown.to_csv(PROJECT_ROOT / 'data/processed/aqi_category_breakdown.csv', index=False)

# 4. Annual PM2.5 Trend (2010–2014)
annual_pm25_trend = df.groupby('year')['pm2_5_cleaned'].mean().reset_index(name='avg_pm25')
annual_pm25_trend.to_csv(PROJECT_ROOT / 'data/processed/annual_pm25_trend.csv', index=False)

# Dashboard 3 — Weather & Conditions

# 1. PM2.5 by Wind Direction (Calm vs NW)
# Assuming 'calm' is a category, but in data it's directions. Perhaps filter for 'NW' and 'calm' if exists.
# From EDA, wind_direction has various, but for Calm vs NW, perhaps mean for NW and for calm.
# But calm might not be there. Let's compute avg by wind_direction, and note.
pm25_by_wind_dir = df.groupby('wind_direction')['pm2_5_cleaned'].mean().reset_index(name='avg_pm25')
pm25_by_wind_dir.to_csv(PROJECT_ROOT / 'data/processed/pm25_by_wind_direction.csv', index=False)

# 2. PM2.5 by Wind Speed (binned ranges)
bins = [0, 1, 2, 3, 4, 5, 10, 20]  # example bins
labels = ['0-1', '1-2', '2-3', '3-4', '4-5', '5-10', '10-20']
df['wind_speed_bin'] = pd.cut(df['wind_speed'], bins=bins, labels=labels, right=False)
pm25_by_wind_speed_bin = df.groupby('wind_speed_bin')['pm2_5_cleaned'].mean().reset_index(name='avg_pm25')
pm25_by_wind_speed_bin.to_csv(PROJECT_ROOT / 'data/processed/pm25_by_wind_speed_bins.csv', index=False)

# 3. Seasonal Mean PM2.5 vs WHO Limit
df['season'] = df['month'].map({12:'Winter',1:'Winter',2:'Winter',
                                 3:'Spring',4:'Spring',5:'Spring',
                                 6:'Summer',7:'Summer',8:'Summer',
                                 9:'Autumn',10:'Autumn',11:'Autumn'})
seasonal_mean_pm25 = df.groupby('season')['pm2_5_cleaned'].mean().reset_index(name='avg_pm25')
seasonal_mean_pm25['who_limit'] = 15
seasonal_mean_pm25.to_csv(PROJECT_ROOT / 'data/processed/seasonal_mean_pm25_vs_who.csv', index=False)

# 4. Precipitation Effect on PM2.5 (Rain vs Snow vs Dry)
df['precip_type'] = 'Dry'
df.loc[df['hours_rain'] > 0, 'precip_type'] = 'Rain'
df.loc[df['hours_snow'] > 0, 'precip_type'] = 'Snow'
precip_effect = df.groupby('precip_type')['pm2_5_cleaned'].mean().reset_index(name='avg_pm25')
precip_effect.to_csv(PROJECT_ROOT / 'data/processed/precipitation_effect_on_pm25.csv', index=False)

In [ ]:
precip_effect.to_csv(PROJECT_ROOT / 'data/processed/precipitation_effect_on_pm25.csv', index=False)

TABLEAU_READY_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(TABLEAU_READY_PATH, index=False)
print(f'Saved Tableau-ready dataset to {TABLEAU_READY_PATH}')